# Salary Prediction Project

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Data Overview

In [2]:
salary_df = pd.read_csv('../data/data.csv')

In [3]:
salary_df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0


In [4]:
salary_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6704 entries, 0 to 6703
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Age                  6702 non-null   float64
 1   Gender               6702 non-null   str    
 2   Education Level      6701 non-null   str    
 3   Job Title            6702 non-null   str    
 4   Years of Experience  6701 non-null   float64
 5   Salary               6699 non-null   float64
dtypes: float64(3), str(3)
memory usage: 552.7 KB


In [5]:
print(f'Rows: {salary_df.shape[0]}\nColumns: {salary_df.shape[1]}')

Rows: 6704
Columns: 6


## Data Preprocessing

In [6]:
salary_df.duplicated().sum()

np.int64(4912)

In [7]:
salary_df.isnull().sum()

Age                    2
Gender                 2
Education Level        3
Job Title              2
Years of Experience    3
Salary                 5
dtype: int64

In [8]:
salary_df = salary_df.dropna().reset_index(drop=True)

In [9]:
salary_df.columns = salary_df.columns.str.replace(' ', '_')
print(salary_df.columns)

Index(['Age', 'Gender', 'Education_Level', 'Job_Title', 'Years_of_Experience',
       'Salary'],
      dtype='str')


In [10]:
salary_df.Education_Level = salary_df.Education_Level.str.replace(' ','_').str.lower()
salary_df.Education_Level = salary_df.Education_Level.str.replace('_degree','')
salary_df.Education_Level = salary_df.Education_Level.str.replace("'s",'')
salary_df.Education_Level.value_counts()

Education_Level
bachelor       3021
master         1860
phd            1369
high_school     448
Name: count, dtype: int64

In [11]:
salary_df.Job_Title = salary_df.Job_Title.str.replace(' ','_').str.lower()
salary_df.Job_Title = salary_df.Job_Title.str.replace('juniour','junior')#.str.lower()
salary_df.Job_Title.value_counts()

Job_Title
software_engineer                518
data_scientist                   453
software_engineer_manager        376
data_analyst                     363
senior_project_engineer          318
                                ... 
junior_research_scientist          1
senior_hr_specialist               1
junior_operations_coordinator      1
junior_financial_advisor           1
social_media_man                   1
Name: count, Length: 188, dtype: int64

In [12]:
def group_job_title(title):
    # title = str(title).lower().strip()
    # # fix typo in dataset
    # title = title.replace('juniour', 'junior')

    # 1. Management & Leadership (check first — catches director/manager titles early)
    if any(k in title for k in [
        'ceo', 'cto', 'coo', 'chief', 'vp_of', 'vp of',
        'director', 'senior_manager', 'senior manager',
        'operations_manager', 'software_manager', 'office_manager',
        'software_project_manager', 'senior_project_manager',
        'senior_operations_manager', 'senior_sales_manager',
        'senior_marketing_manager', 'senior_hr_manager',
        'senior_business_development_manager', 'senior_product_development_manager',
        'junior_operations_manager', 'senior_marketing_director',
        'senior_operations_coordinator', 'software_engineer_manager',
        'senior_project_coordinator', 'senior_project_engineer'
    ]):
        return 'Management & Leadership'

    # 2. Software Engineering & IT
    elif any(k in title for k in [
        'software_engineer', 'software engineer', 'full_stack', 'front_end',
        'back_end', 'web_developer', 'software_developer', 'junior_developer',
        'senior_engineer', 'principal_engineer', 'network_engineer',
        'software_architect', 'junior_software', 'senior_software',
        'junior_web_developer', 'it_support', 'it_manager',
        'senior_it', 'junior_it', 'help_desk', 'technical_support',
        'devops', 'cloud', 'data_entry_clerk', 'project_engineer'
    ]):
        return 'Software Engineering & IT'

    # 3. Data & Analytics
    elif any(k in title for k in [
        'data_scientist', 'data_analyst', 'data_engineer',
        'business_analyst', 'business_intelligence', 'junior_data',
        'senior_data', 'supply_chain_analyst', 'operations_analyst',
        'senior_business_analyst', 'junior_business_analyst',
        'senior_operations_analyst', 'junior_operations_analyst',
        'quality_assurance_analyst', 'senior_quality_assurance',
        'chief_data_officer', 'junior_marketing_analyst',
        'senior_marketing_analyst', 'marketing_analyst',
        'financial_analyst', 'junior_financial_analyst',
        'senior_financial_analyst', 'senior_business_operations_analyst'
    ]):
        return 'Data & Analytics'

    # 4. Product & Research
    elif any(k in title for k in [
        'product_manager', 'product_owner', 'junior_product_manager',
        'senior_product_manager', 'product_marketing_manager',
        'senior_product_marketing_manager', 'director_of_product',
        'research_scientist', 'senior_research_scientist',
        'junior_research_scientist', 'principal_scientist',
        'senior_scientist', 'senior_researcher', 'researcher',
        'ux_researcher', 'research_director'
    ]):
        return 'Product & Research'

    # 5. Marketing, Content & Design
    elif any(k in title for k in [
        'marketing', 'social_media', 'content', 'copywriter',
        'graphic_designer', 'ux_designer', 'junior_ux_designer',
        'senior_ux_designer', 'junior_web_designer', 'web_designer',
        'product_designer', 'senior_product_designer', 'junior_designer',
        'creative_director', 'advertising', 'seo', 'brand',
        'public_relations', 'digital_content', 'technical_writer',
        'event_coordinator'
    ]):
        return 'Marketing & Design'

    # 6. Sales & Business Development
    elif any(k in title for k in [
        'sales', 'account_manager', 'junior_account_manager',
        'senior_account_manager', 'senior_account_executive',
        'business_development', 'junior_business_development',
        'customer_success', 'sales_executive', 'sales_associate',
        'strategy_consultant', 'senior_consultant', 'consultant'
    ]):
        return 'Sales & Business Development'

    # 7. Finance & Accounting
    elif any(k in title for k in [
        'financial_manager', 'accountant', 'junior_accountant',
        'senior_accountant', 'financial_advisor', 'junior_financial_advisor',
        'senior_financial_advisor', 'senior_financial_manager',
        'finance', 'vp_of_finance'
    ]):
        return 'Finance & Accounting'

    # 8. HR & Recruiting
    elif any(k in title for k in [
        'hr', 'human_resources', 'human resources', 'recruiter',
        'recruiting', 'training_specialist', 'senior_training_specialist',
        'talent', 'people_ops'
    ]):
        return 'HR & Recruiting'

    # 9. Operations & Admin
    elif any(k in title for k in [
        'operations', 'supply_chain', 'administrative',
        'receptionist', 'coordinator', 'project_manager',
        'project_coordinator', 'junior_project_manager',
        'office_manager', 'account_executive', 'delivery_driver',
        'customer_service', 'customer service', 'support_specialist',
        'customer_support', 'service_rep', 'service_manager',
        'service_representative'
    ]):
        return 'Operations & Admin'

    else:
        return 'Other'

In [13]:
salary_df['Job_Group'] = salary_df['Job_Title'].apply(group_job_title)
salary_df['Job_Group'] = salary_df['Job_Group'].str.replace(" ", "_")
print(salary_df['Job_Group'].value_counts())
print("\nTitles in 'Other':")
print(salary_df[salary_df['Job_Group'] == 'Other']['Job_Title'].unique())

Job_Group
Software_Engineering_&_IT       1969
Management_&_Leadership         1557
Data_&_Analytics                1140
Marketing_&_Design               589
Product_&_Research               518
Sales_&_Business_Development     404
HR_&_Recruiting                  269
Finance_&_Accounting             150
Operations_&_Admin               102
Name: count, dtype: int64

Titles in 'Other':
<ArrowStringArray>
[]
Length: 0, dtype: str


In [14]:
salary_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6698 entries, 0 to 6697
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Age                  6698 non-null   float64
 1   Gender               6698 non-null   str    
 2   Education_Level      6698 non-null   str    
 3   Job_Title            6698 non-null   str    
 4   Years_of_Experience  6698 non-null   float64
 5   Salary               6698 non-null   float64
 6   Job_Group            6698 non-null   str    
dtypes: float64(3), str(4)
memory usage: 709.7 KB


In [15]:
salary_df.head(20)

,Age,Gender,Education_Level,Job_Title,Years_of_Experience,Salary,Job_Group
0,32.0,Male,bachelor,software_engineer,5.0,90000.0,Software_Engineering_&_IT
1,28.0,Female,master,data_analyst,3.0,65000.0,Data_&_Analytics
2,45.0,Male,phd,senior_manager,15.0,150000.0,Management_&_Leadership
3,36.0,Female,bachelor,sales_associate,7.0,60000.0,Sales_&_Business_Development
4,52.0,Male,master,director,20.0,200000.0,Management_&_Leadership
5,29.0,Male,bachelor,marketing_analyst,2.0,55000.0,Data_&_Analytics
6,42.0,Female,master,product_manager,12.0,120000.0,Product_&_Research
7,31.0,Male,bachelor,sales_manager,4.0,80000.0,Sales_&_Business_Development
8,26.0,Female,bachelor,marketing_coordinator,1.0,45000.0,Management_&_Leadership
9,38.0,Male,phd,senior_scientist,10.0,110000.0,Product_&_Research


## Modeling

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

In [18]:
X = salary_df.drop(columns=['Age', 'Gender', 'Job_Title', 'Salary'])
y = salary_df['Salary']

In [19]:
numeric_features = ['Years_of_Experience']
ordinal_features = ['Education_Level']
object_features = ['Job_Group']

In [20]:
education_order = [['high_school', 'bachelor', 'master', 'phd']]

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric_transform', StandardScaler(), numeric_features),
        ('ordinal_transform', OrdinalEncoder(categories=education_order), ordinal_features),
        ('object_transform', OneHotEncoder(handle_unknown='ignore'), object_features),
    ]
)

In [22]:
from sklearn.model_selection import train_test_split

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,) #stratify=y)

In [24]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

In [25]:
lr_model = Pipeline(steps=[('preprocess', preprocessor), ('LinearRegression', LinearRegression())])
dt_model = Pipeline(steps=[('preprocess', preprocessor), ('DecisionTree', DecisionTreeRegressor(random_state=42))])
rf_model = Pipeline(steps=[('preprocess', preprocessor), ('RandomForest', RandomForestRegressor(random_state=42))])
knn_model = Pipeline(steps=[('preprocess', preprocessor), ('KNN', KNeighborsRegressor())])

In [26]:
lr_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('LinearRegression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_transform', ...), ('ordinal_transform', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If t

In [27]:
dt_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('DecisionTree', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_transform', ...), ('ordinal_transform', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the o

In [28]:
rf_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('RandomForest', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_transform', ...), ('ordinal_transform', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the o

In [29]:
knn_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('KNN', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_transform', ...), ('ordinal_transform', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of 

In [30]:
lr_y = lr_model.predict(X_test)
dt_y = dt_model.predict(X_test)
rf_y = rf_model.predict(X_test)
knn_y = knn_model.predict(X_test)

In [31]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

In [32]:
lr_eval = [mean_absolute_error(y_test, lr_y), mean_squared_error(y_test, lr_y), root_mean_squared_error(y_test, lr_y), r2_score(y_test, lr_y)]
dt_eval = [mean_absolute_error(y_test, dt_y), mean_squared_error(y_test, dt_y), root_mean_squared_error(y_test, dt_y), r2_score(y_test, dt_y)]
rf_eval = [mean_absolute_error(y_test, rf_y), mean_squared_error(y_test, rf_y), root_mean_squared_error(y_test, rf_y), r2_score(y_test, rf_y)]
knn_eval = [mean_absolute_error(y_test, knn_y), mean_squared_error(y_test, knn_y), root_mean_squared_error(y_test, knn_y), r2_score(y_test, knn_y)]

In [33]:
for i in range(4):
    lr_eval[i] = np.round(lr_eval[i], 2)
    dt_eval[i] = np.round(dt_eval[i], 2)
    rf_eval[i] = np.round(rf_eval[i], 2)
    knn_eval[i] = np.round(knn_eval[i], 2)

In [34]:
table_performance = pd.DataFrame({
    'Models': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'KNN'],
    'MAE': [lr_eval[0], dt_eval[0], rf_eval[0], knn_eval[0]],
    'MSE': [lr_eval[1], dt_eval[1], rf_eval[1], knn_eval[1]],
    'RMSE': [lr_eval[2], dt_eval[2], rf_eval[2], knn_eval[2]],
    'R2-Score': [lr_eval[3], dt_eval[3], rf_eval[3], knn_eval[3]],
})

table_performance

,Models,MAE,MSE,RMSE,R2-Score
0,Logistic Regression,21189.84,7.519485e+08,27421.68,0.74
1,Decision Tree,11863.39,3.409763e+08,18465.54,0.88
2,Random Forest,11768.74,3.240282e+08,18000.78,0.89
3,KNN,12809.30,3.898718e+08,19745.17,0.86


In [35]:
table_performance = pd.DataFrame({
    'Models': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'KNN'],
    'MAE': [lr_eval[0], dt_eval[0], rf_eval[0], knn_eval[0]],
    'MSE': [lr_eval[1], dt_eval[1], rf_eval[1], knn_eval[1]],
    'RMSE': [lr_eval[2], dt_eval[2], rf_eval[2], knn_eval[2]],
    'R2-Score': [lr_eval[3], dt_eval[3], rf_eval[3], knn_eval[3]],
})

table_performance

,Models,MAE,MSE,RMSE,R2-Score
0,Logistic Regression,21189.84,7.519485e+08,27421.68,0.74
1,Decision Tree,11863.39,3.409763e+08,18465.54,0.88
2,Random Forest,11768.74,3.240282e+08,18000.78,0.89
3,KNN,12809.30,3.898718e+08,19745.17,0.86


## Fine-tuning

In [36]:
from sklearn.model_selection import RandomizedSearchCV

In [37]:
param = {
    'RandomForest__n_estimators':[100, 200, 300],
    'RandomForest__max_depth':[None, 5, 10, 15],
    'RandomForest__min_samples_split':[2, 5, 10], 
    'RandomForest__min_samples_leaf':[1, 2, 4]
}

In [38]:
search = RandomizedSearchCV(rf_model, 
                            param_distributions=param, 
                            cv=5, 
                            scoring='r2',
                            n_iter=20,
                            random_state=42)

In [39]:
search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'RandomForest__max_depth': [None, 5, ...], 'RandomForest__min_samples_leaf': [1, 2, ...], 'RandomForest__min_samples_split': [2, 5, ...], 'RandomForest__n_estimators': [100, 200, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variou

In [40]:
print('Best params:', search.best_params_)
print('Best CV R2', search.best_score_)

Best params: {'RandomForest__n_estimators': 200, 'RandomForest__min_samples_split': 5, 'RandomForest__min_samples_leaf': 1, 'RandomForest__max_depth': None}
Best CV R2 0.8837881403339185


In [41]:
best_model = search.best_estimator_
best_y = best_model.predict(X_test)

In [42]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2')
print(f"CV R² scores: {scores}")
print(f"Mean R²: {scores.mean():.4f}")
print(f"Std: {scores.std():.4f}")

CV R² scores: [0.8691134  0.88185986 0.89661755 0.88856157 0.88278833]
Mean R²: 0.8838
Std: 0.0090


In [43]:
best_eval = [mean_absolute_error(y_test, best_y), 
             mean_squared_error(y_test, best_y), 
             root_mean_squared_error(y_test, best_y), 
             r2_score(y_test, best_y)]

for i in range(4):
    best_eval[i] = np.round(best_eval[i], 2)

In [44]:
new_table_performance = pd.DataFrame({
    'Models': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'KNN', 'Best Model'],
    'MAE': [lr_eval[0], dt_eval[0], rf_eval[0], knn_eval[0], best_eval[0]],
    'MSE': [lr_eval[1], dt_eval[1], rf_eval[1], knn_eval[1], best_eval[1]],
    'RMSE': [lr_eval[2], dt_eval[2], rf_eval[2], knn_eval[2], best_eval[2]],
    'R2-Score': [lr_eval[3], dt_eval[3], rf_eval[3], knn_eval[3], best_eval[3]],
})

new_table_performance

,Models,MAE,MSE,RMSE,R2-Score
0,Logistic Regression,21189.84,7.519485e+08,27421.68,0.74
1,Decision Tree,11863.39,3.409763e+08,18465.54,0.88
2,Random Forest,11768.74,3.240282e+08,18000.78,0.89
3,KNN,12809.30,3.898718e+08,19745.17,0.86
4,Best Model,11792.22,3.241031e+08,18002.86,0.89


### Testing With Unknown Data

In [45]:
sample = pd.DataFrame({
    'Education_Level':["bachelor"],
    'Job_Group':['Data_&_Analytics'],
    'Years_of_Experience':[3.0],
})

In [46]:
print('Lr model', lr_model.predict(sample)[0])
print('Dt model', dt_model.predict(sample)[0])
print('rf model', rf_model.predict(sample)[0])
print('knn model', knn_model.predict(sample)[0])

print(f"Predicted Value: {best_model.predict(sample)[0]:.2f}")

Lr model 93836.072140296
Dt model 94878.5046728972
rf model 95165.31720131842
knn model 90000.0
Predicted Value: 95211.58


Exporting

In [47]:
import joblib

In [48]:
joblib.dump(best_model, '../models/model3.pkl')
print('Model has been successfully saved!')

Model has been successfully saved!


In [17]:
salary_df = salary_df.drop(columns=['Age', 'Gender', 'Job_Title'])

In [18]:
salary_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6698 entries, 0 to 6697
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Education_Level      6698 non-null   str    
 1   Years_of_Experience  6698 non-null   float64
 2   Salary               6698 non-null   float64
 3   Job_Group            6698 non-null   str    
dtypes: float64(2), str(2)
memory usage: 394.4 KB


In [20]:
salary_df.to_csv('../data/cleaned_data.csv', index=False)